# Midterm 1: run the data generator in Colab

Use this notebook if you cannot run Python or git on your own machine. It only **generates and shows your data**. You write your answers in `ANSWERS.md` on **your fork on github.com** (open the file, click the pencil icon, edit, *Commit changes*), then submit the commit URL in Canvas. See the README for the rules.

1. Fill in `GITHUB_USERNAME` (the account that owns your fork) and `CODE` (`PRACTICE` to rehearse; the real code is on the board during the exam).
2. Run the three code cells in order.

In [2]:
GITHUB_USERNAME = "tomiris-nurbolat"   # e.g. "aigerim-k"
CODE = "PRACTICE"      # replace with the session code during the exam
assert GITHUB_USERNAME, "fill in GITHUB_USERNAME first"

In [3]:
import os

if not os.path.exists("midterm-1-AI-course"):
    !git clone --quiet https://github.com/unreal-kz/midterm-1-AI-course.git
%cd midterm-1-AI-course
!python make_my_data.py --github {GITHUB_USERNAME} --code {CODE}

/content/midterm-1-AI-course
Wrote 3 files to my_data/ · fingerprint bfd9a65e
Next: open my_data/DATA.md


In [4]:
import pandas as pd
from IPython.display import Markdown, display

display(Markdown(open("my_data/DATA.md", encoding="utf-8").read()))
ranks = pd.read_csv("my_data/retrieval_ranks.csv")   # Part 1
docs = pd.read_csv("my_data/doc_results.csv")        # Part 2
display(ranks.head(), docs)

# Midterm 1: your data

GitHub username: `tomiris-nurbolat` · session code: `practice` · generator v1

**Fingerprint: `bfd9a65e`**: copy it into the first lines of `ANSWERS.md`. All numbers below are synthetic: this is an exam, not a measurement.

## Part 1: inputs

`retrieval_ranks.csv`: 60 test questions (20 EN, 20 RU, 20 KK). `rank_A` and `rank_B` give the position of the correct chunk in the top 10 retrieved under config A and config B; **11 means the correct chunk was not in the top 10.**

Production traffic and Model M token counts (tokens per item, as counted by Model M's tokenizer):

| language | share of traffic | question tokens | answer tokens | chunk A tokens | chunk B tokens |
|---|---|---|---|---|---|
| EN | 26 % | 32 | 144 | 264 | 79 |
| RU | 35 % | 73 | 327 | 599 | 179 |
| KK | 39 % | 100 | 448 | 821 | 246 |

- Requests per day: **6,500**, in the traffic shares above.
- System prompt: **194** tokens in every request.
- Price: **$2** per million input tokens, **$12** per million output tokens.
- Annual budget for this service (model calls only): **$21,900**.

## Part 2: inputs

`doc_results.csv`: for each pipeline, language and scan condition, `n_docs` test invoices and `n_correct` = invoices whose number, date, seller ID and total were all extracted exactly (strict scorer, as in Lab 04).

- **Pipeline H**: hosted vision-language API in a data centre abroad, pay per token; each page is sent as an image.
- **Pipeline L**: open vision-language model on your own server in Kazakhstan, fixed monthly cost.

- Volume: **5,500** scanned pages per day. In **9 %** of them an individual's name and ИИН are printed (sole proprietors, receipts).
- Page image: **827 × 1169** pixels. Pipeline H counts an image as ⌈width ÷ 28⌉ × ⌈height ÷ 28⌉ input tokens and returns about **359** output tokens per page (the JSON).
- Pipeline H price: **$2** per million input tokens, **$10** per million output tokens.
- Pipeline L: one server costs **$900** per month and handles up to **6,000** pages per day; add servers as needed.

## Part 3: your five questions

Answer each in `ANSWERS.md`: one **Number** line and one **Why** line of at most 40 words.

### Q1

The same support request, once in English and once in Kazakh, through Model M ($1 per million input tokens, $5 per million output tokens):

| | input tokens | output tokens |
|---|---|---|
| EN | 1,796 | 105 |
| KK | 4,843 | 210 |

Compute the token ratio (total tokens KK ÷ EN) and the bill ratio (cost KK ÷ cost EN). **Number:** the bill ratio (2 decimals). **Why** (at most 40 words): why is the bill ratio not equal to the token ratio?

### Q2

A teammate sets the chunk size to **512 tokens, counted with cl100k**. The retrieval model (e5-small) reads at most 512 of *its own* tokens. Assume e5 tokens grow in proportion to cl100k tokens at the rate measured in Lecture 4.1 (a chunker told 256 cl100k tokens produced chunks of 263 / 128 / 76 e5 tokens for EN / RU / KK), and that every chunk is exactly the full size.

**Number:** the share of an English chunk, in percent (1 decimal), that the embedding never sees. **Why** (at most 40 words): which of the three languages are cut, and what happens to the cut part?

### Q3

A language model gives these next-token scores for the prompt "The branch is closed on":

| token | score |
|---|---|
| Monday | 4.2 |
| Friday | 2.8 |
| Sunday | 2.3 |
| holiday | 1.3 |
| Easter | 0.9 |

Apply softmax at temperature T = 0.5. **Number:** the probability of the top token, in percent (1 decimal). **Why** (at most 40 words): what did changing T do to the ranking of the tokens, and to the set of tokens that can be sampled?

### Q4

Two models answer the same 100 factual questions (1 point per correct answer on the public leaderboard):

| | correct | wrong | "I don't know" |
|---|---|---|---|
| Model X | 70 | 30 | 0 |
| Model Y | 64 | 24 | 12 |

Now score them again with a penalty: +1 per correct answer, −1 per wrong answer, 0 for "I don't know". **Number:** Y's penalised score minus X's (negative if X is ahead). **Why** (at most 40 words): which model does the public leaderboard prefer, and what does that teach a model to do?

### Q5

A support chat adds **95 new tokens per turn** (customer message plus assistant reply) and has **23 turns**. The model keeps no state, so the full history is sent again on every turn (Lecture 1.2). As in the Lecture 1.2 table (100, 200, 300, ... tokens billed on turns 1, 2, 3, ...), turn *i* is billed for the tokens of all *i* turns so far, including its own. **Number:** the total input tokens billed over the whole conversation. **Why** (at most 40 words): how many times has the first message been paid for by the last turn, and what does prompt caching change?


,qid,lang,rank_A,rank_B
0,q01,EN,1,1
1,q02,KK,1,1
2,q03,EN,1,1
3,q04,RU,11,11
4,q05,EN,1,1


,pipeline,lang,condition,n_docs,n_correct
0,H,EN,clean,30,29
1,H,EN,degraded,30,27
2,H,RU,clean,30,28
3,H,RU,degraded,30,26
4,H,KK,clean,30,27
5,H,KK,degraded,30,25
6,L,EN,clean,30,27
7,L,EN,degraded,30,23
8,L,RU,clean,30,26
9,L,RU,degraded,30,21


`ranks` and `docs` are ready for you to compute with. Remember the **fingerprint** at the top of the data sheet: it goes into the first lines of `ANSWERS.md`.